In [1]:
import math
from pathlib import Path

import openmc

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import LogNorm
from openmc.deplete import d1s
from openmc.data.data import half_life
import structural_material_analysis as sma
import tokamak_radiation_environment as tre

/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/material.py:1512: UserWarning: Warning: sum of fractions do not add to 1, void fraction set to 0.09999999999999987
  warnings.warn(msg)


In [2]:
openmc.config['chain_file'] = r'/home/davide/openmc_models/CROSS_SECTIONS/decay/endfb-viii.0-chain/chain_endfb80_sfr.xml'

In [3]:
inner_vessel_material = sma.materials.inconel718

In [4]:
# Volumes of regions expressed in the unit of cm3 (from SolidWorks)
# Volumes of real geometry
INNER_volume = 181437118.09
FW_volume = 352170.38
STR1_volume = 3529943.32
FLiBe1_volume = 7104771.83
NM_volume = 3574791.48
STR2_volume = 10813774.60
FLiBe2_volume = 350171938.29
STR3_volume = 14002357.77

fusion_power = 525e6  # units Watts
energy_per_fusion_reaction = 17.6e6  # units eV
eV_to_Joules = 1.60218e-19  # multiplication factor to convert eV to Joules
number_of_neutrons_per_second = fusion_power / (energy_per_fusion_reaction * eV_to_Joules)
print('Number of neutrons per second', number_of_neutrons_per_second, '\n')

Number of neutrons per second 1.8618098749544654e+20 



In [5]:
inner_vessel_material.volume = STR1_volume

In [6]:
dt_plasma = sma.materials.dt_plasma
tungsten = sma.materials.tungsten
flibe = sma.materials.flibe
beryllium = sma.materials.beryllium
boroncarbide = sma.materials.boroncarbide
inconel718 = sma.materials.inconel718

materials = openmc.Materials([
    dt_plasma,
    tungsten,
    inner_vessel_material,
    flibe, 
    boroncarbide, 
    beryllium, 
    ])

(root, geometry) = sma.geometry.reactor_geometry(dt_plasma, tungsten, inner_vessel_material, flibe, beryllium, inconel718, boroncarbide)

############################################################################
# Define Settings
source = openmc.IndependentSource()
source.particle = "neutron"
radius = openmc.stats.Discrete([285], [1])
z_values = openmc.stats.Discrete([0], [1])
angle = openmc.stats.Uniform(a=np.radians(-10), b=np.radians(10))
source.space = openmc.stats.CylindricalIndependent(
    r=radius, phi=angle, z=z_values, origin=(0.0, 0.0, 0.0)
)
source.angle = openmc.stats.Isotropic()
source.energy = openmc.stats.muir(e0=14080000.0, m_rat=5.0, kt=20000.0)

settings = openmc.Settings()
settings.batches = 100
settings.particles = 10000
settings.inactive = 0
settings.run_mode = "fixed source"
settings.source = source
settings.photon_transport = True  # This line is required to switch on photons tracking
settings.output = {"tallies": False}

# D1S specific setting
settings.use_decay_photons = True

model = openmc.Model(geometry, materials, settings)

In [ ]:
# creates a regular mesh that surrounds the geometry for the tally
# mesh = openmc.RegularMesh().from_domain(
#     geometry,
#     dimension=[1000, 1, 1000],
#     # 100 voxels in x and y axis directions and 1 voxel in z as we want a xy plot
# )

mesh = openmc.RegularMesh()
mesh.dimension = [300, 1, 200]
mesh.lower_left = [100, -100, -250]  # x,y,z coordinates start at 0 as this is a sector model
mesh.upper_right = [500, 0, 250]
mesh_filter = openmc.MeshFilter(mesh)  # creating a mesh
    
# adding a dose tally on a regular mesh
# AP, PA, LLAT, RLAT, ROT, ISO are ICRP incident dose field directions, AP is front facing
energies, pSv_cm2 = openmc.data.dose_coefficients(particle="photon", geometry="AP")
dose_filter = openmc.EnergyFunctionFilter(
    energies, pSv_cm2, interpolation="cubic"  # interpolation method recommended by ICRP
)
particle_filter = openmc.ParticleFilter(["photon"])
mesh_filter = openmc.MeshFilter(mesh)
str1_filter = openmc.CellFilter(root[3])
dose_tally = openmc.Tally()
dose_tally.filters = [particle_filter, mesh_filter, dose_filter, str1_filter]
dose_tally.scores = ["flux"]
dose_tally.name = "photon_dose_on_mesh"

tallies = openmc.Tallies([dose_tally])

In [8]:
model = openmc.Model(geometry, materials, settings, tallies)

# this adds ParentNuclideFilter to each tally, which the D1S method requires
d1s.prepare_tallies(model=model)

output_path = model.run()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

RuntimeError: std::bad_alloc

In [ ]:
# Get tally from statepoint
with openmc.StatePoint(output_path) as sp:
    dose_tally_from_sp = sp.get_tally(name='photon_dose_on_mesh')

In [ ]:
timesteps_and_source_rates = [
    (1, 1e18),  # 1 second
    (60*20, 0),  # 20 minutes
    (60*20, 0),  # 40 minutes
    (60*20, 0),  # 60 minutes
    (60*20, 0),  # 80 minutes
    (60*20, 0),  # 100 minutes
]

timesteps = [item[0] for item in timesteps_and_source_rates]
source_rates = [item[1] for item in timesteps_and_source_rates]

# this gets all the unstable nuclides that can be produced during D1S
radionuclides = d1s.get_radionuclides(model)

# Compute time correction factors based on irradiation schedule
time_factors = d1s.time_correction_factors(
    nuclides=radionuclides,
    timesteps=timesteps,
    source_rates=source_rates,
    timestep_units = 's'
)

In [ ]:
for i_cool in range(1, len(timesteps)):  # missing the first timestep as it is the irradiation step

    # Apply time correction factors
    # this includes the source_rates which are in units of neutrons per second
    # dose_tally_from_sp is in units of pSv-cm3/source neutron
    # corrected tally is now in units of pSv-cm3/second
    corrected_tally = d1s.apply_time_correction(
        tally=dose_tally_from_sp,
        time_correction_factors=time_factors,
        index=i_cool,
        sum_nuclides=True
    )

    # multiplication by pico_to_milli converts from (pico) pSv to (milli) mSv
    pico_to_milli = 1e-9
    
    # divided by mesh element volume converts from mSv-cm3 to mSv
    volume_normalization = mesh.volumes[0][0][0]
    
    # this section simply gets the maximum value of the mean tally across all time steps
    # and uses this to set the max value of the color bar in the plots
    if i_cool == 1:
        max_tally_value = max(corrected_tally.mean).flatten()
        scaled_max_tally_value = (max_tally_value * pico_to_milli) / volume_normalization

    # get a slice of mean values on the xy basis mid z axis
    corrected_tally_mean = corrected_tally.get_reshaped_data(value='mean', expand_dims=True).squeeze()
    # create a plot of the mean flux values
    
    scaled_corrected_tally_mean = (corrected_tally_mean * pico_to_milli) / volume_normalization
    
    fig, ax1 = plt.subplots(figsize=(6, 4))
    plot_1 = ax1.imshow(
        X=scaled_corrected_tally_mean.T,
        origin="lower",
        extent=mesh.bounding_box.extent['xy'],
        norm=LogNorm(vmax=scaled_max_tally_value)
    )

    ax2 = geometry.plot(
        outline='only',
        extent=geometry.bounding_box.extent['xy'],
        axes=ax1,  # Use the same axis as ax1
        pixels=10_000_00,  #avoids rounded corners on outline
    )

    time_in_mins = round(sum(timesteps[1:i_cool+1])/(60),2)
    max_dose_in_timestep = round(max(scaled_corrected_tally_mean.flatten()), 2)

    ax2.set_title(f"Dose Rate at time {time_in_mins} minutes after irradiation\nMax dose rate: {max_dose_in_timestep} mSv/s")
    ax2.set_xlim(ax1.get_xlim())
    ax2.set_ylim(ax1.get_ylim())
    ax2.set_aspect(ax1.get_aspect())  # Match aspect ratio
    ax2.set_xlabel("X (cm)")
    ax2.set_ylabel("Y (cm)")
    cbar = plt.colorbar(plot_1, ax=ax1)
    cbar.set_label("Dose [milli Sv per second]")  # Label for the color bar

    plt.show()
    plt.close()

In [ ]:
# Define one irradiation step and one cooling step
irradiation_step = (1, 1e18)
cooling_step = (60 * 5, 0)  # 5 mins of cooling with no source rate

# Build one cycle: irradiation + N cooling steps
cooling_steps = [cooling_step] * 24
pulse_cycle = [irradiation_step] + cooling_steps

# Repeat the cycle 5 times
timesteps_and_source_rates = pulse_cycle * 4
timesteps = [item[0] for item in timesteps_and_source_rates]
source_rates = [item[1] for item in timesteps_and_source_rates]

# this gets all the unstable nuclides that can be produced during D1S
radionuclides = d1s.get_radionuclides(model)

# Compute time correction factors based on irradiation schedule
time_factors = d1s.time_correction_factors(
    nuclides=radionuclides,
    timesteps=timesteps,
    source_rates=source_rates,
    timestep_units = 's'
)

scaled_max_tally_values = []
for i_cool in range(1, len(timesteps)):  # missing the first timestep as it is the irradiation step

    # Apply time correction factors
    # this includes the source_rates which are in units of neutrons per second
    # dose_tally_from_sp is in units of pSv-cm3/source neutron
    # corrected tally is now in units of pSv-cm3/second
    corrected_tally = d1s.apply_time_correction(
        tally=dose_tally_from_sp,
        time_correction_factors=time_factors,
        index=i_cool,
        sum_nuclides=True
    )

    # this section simply gets the maximum value of the mean tally across all time steps
    scaled_max_tally_values.append((max(corrected_tally.mean).flatten()* pico_to_milli) / volume_normalization)

In [ ]:
plt.plot(np.cumsum(timesteps[1:]), scaled_max_tally_values)
plt.xlabel("Time (s)")
plt.ylabel("Max Dose Rate (mSv/s)")
plt.title("Max Dose Rate Over Time")
plt.grid()
plt.show()

In [ ]:
corrected_tally = d1s.apply_time_correction(
    tally=dose_tally_from_sp,
    time_correction_factors=time_factors,
    index=i_cool,
    sum_nuclides=False
)

# one can get the nuclides from ealier in the code
# In this case we get the nuclides from the tally.
# This allows us to full detach the post processing from the simulation script.
# This is useful if you want to post process a statepoint file and not have the script that made the simulation.
parent_nuclide_filter = corrected_tally.find_filter(openmc.ParentNuclideFilter)
parent_nuclides = parent_nuclide_filter.bins


scaled_max_tally_values_per_nuclide = {str(nuclide): [] for nuclide in parent_nuclides}

for i_cool in range(1, len(timesteps)):  # missing the first timestep as it is the irradiation step

    # Apply time correction factors
    # this includes the source_rates which are in units of neutrons per second
    # dose_tally_from_sp is in units of pSv-cm3/source neutron
    # corrected tally is now in units of pSv-cm3/second
    corrected_tally = d1s.apply_time_correction(
        tally=dose_tally_from_sp,
        time_correction_factors=time_factors,
        index=i_cool,
        sum_nuclides=False  # to get the individual nuclides
    )
    
    mean_values_per_nuclide = corrected_tally.mean.squeeze().reshape(len(parent_nuclides), -1)
    
    for i_nuclide, nuclide in enumerate(parent_nuclides):

        # this section simply gets the maximum value of the mean tally across all time steps
        max_value_for_nuclide = (max(mean_values_per_nuclide[i_nuclide]).flatten()* pico_to_milli) / volume_normalization
        

        scaled_max_tally_values_per_nuclide[str(nuclide)].append(float(max_value_for_nuclide[0]))

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot(np.cumsum(timesteps[1:]), scaled_max_tally_values, label='total')

for i, nuclide in enumerate(parent_nuclides):
    nuclide_str = str(nuclide)
    if sum(scaled_max_tally_values_per_nuclide[str(nuclide)]) > 2.0:
        plt.plot(np.cumsum(timesteps[1:]), scaled_max_tally_values_per_nuclide[nuclide_str], label=f"{nuclide_str} half-life={half_life(nuclide_str):.1e}s")
plt.legend()
plt.xlabel("Time (s)")
plt.yscale('log')
# plt.xscale('log')
plt.ylabel("Max Dose Rate (mSv/s)")
plt.title("Max Dose Rate Over Time")
plt.grid()
plt.show()

ho ripreso il primo episodio e l'ho finito.
promette bene